
# Outlier-Based Biomarker Discovery for Pregnancy Complications
 Objective: Identify blood, urine, or gut/vaginal microbiome analytes that show extreme deviation in pregnancy complications relative to term controls,  prioritizing these with presistent elevation across multiple timepoints for potential diagnostic biomarker development. 


### Step 1: Calculate Reference Distribution (per analyte)
- for each tissue x timepoint x data type combination
#### 1.1 Subset to term controls only
#### 1.2 Calculate per-analyte statistics
   - Median
   - median avsolute deviation 
       - if control_MAD = 0, flag as "zero_variance" in log and exclude from outlier analysis
#### 1.3 Document control statistics
   - Create reference table with columns as tissue, timepoint, datatype, analyte ID, control median, control_MAD
   - output: control_reference_statistics_tissue_timepoint_datatype.csv

In [63]:
import pandas as pd 
import numpy as np
import seaborn as sns
from scipy import stats

In [13]:
# not all timepoint files are present in all batches!
a1 = pd.read_csv("/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma/Samples_110123_A.csv", index_col=0)
a2 = pd.read_csv("/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma/Samples_51223_A.csv", index_col=0)
a3 = pd.read_csv("/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma/Samples_112524_A.csv", index_col=0)


In [14]:
controls = {}
controls["a"] = pd.concat([a1.loc[list(a1["group"] == "Control"),:].drop(columns=["group", "subgroup", "gestational_age", "gestational_age_at_collection"]), a2.loc[list(a2["group"] == "Control"),:].drop(columns=["group", "subgroup", "gestational_age", "gestational_age_at_collection"]), a3.loc[list(a3["group"] == "Control"),:].drop(columns=["group", "subgroup", "gestational_age", "gestational_age_at_collection"])])


In [8]:
len(a1.columns) - 5

1887

In [106]:
a_med = controls["a"].median()

TypeError: Cannot perform reduction 'median' with string dtype

In [5]:
a_mad = stats.median_abs_deviation(controls["a"])


In [6]:
x = pd.DataFrame(index=a_med.index)

In [7]:
x["med"] = a_med
x["mad"] = a_mad
x["tissue"] = "plasma"
x["timepoint"] = "A"
x["datatype"] = "metabolites"
x["analyte_ID"] = a_med.index

In [135]:
x["D3-Alanine-ISTD_POS"]

KeyError: 'D3-Alanine-ISTD_POS'

In [9]:
plasma_110123 = pd.DataFrame(columns=["tissue", "timepoint", "datatype", "analyte_ID", "control_median", "control_MAD"])
# tissue, timepoint, data type, analyte ID, control median, control MAD

In [10]:
timepoints = ["a", "b", "c", "d", "e"]
for t in timepoints:
    med = controls[t].median()
    mad = stats.median_abs_deviation(controls[t])
    temp = pd.DataFrame()
    temp["control_median"] = med
    temp["control_MAD"] = mad
    temp["tissue"] = "plasma"
    temp["timepoint"] = t
    temp["datatype"] = "metabolites"
    temp["analyte_ID"] = med.index
    temp.to_csv("/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma/control_reference_statistics_plasma_" + t + "_MTBL.csv")
    

KeyError: 'b'

In [124]:
batches = pd.read_csv("/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma/pos_batch.csv")["batch"].unique()

In [125]:
batches = batches.tolist()

In [126]:
x["mad"] == 0

Amoxicillin_POS    False
p487_POS           False
n304_NEG           False
p3014_POS          False
n694_NEG           False
                   ...  
p5751_POS          False
p3224_POS          False
p3083_POS          False
p411_POS           False
p4768_POS          False
Name: mad, Length: 1885, dtype: bool

In [ ]:
# Step 2: Calculate MAD Scores (All Samples)
# For each sample in each tissue at each timepoint for each data type:
# 2.1 Calculate MAD Score
#   For each analyte measurement:
#       MAD_score = (observed_value - Control_Median) / Control_MAD
#   Use the control_median and control_MAD from the appropriate tissue/timepoint/data_type combination
# 2.2 Create MAD score matrix
#   Generate a data structure with:
#       Rows = Sample_ID
#       Columns = Analytes
#       Values = MAD scores
#       Additional metadata = patient_ID, group (control/FGR/GHDP/sPTB), group_subtype, gestational_age, gestational_age_at_sample_collection
# Output: mad_scores_matrix_<tissue>_<timepoint>_<data_type>.csv

In [127]:
def mergeBatches(batches, dir_input, t):
    temp = pd.DataFrame()
    for b in batches:
        try:
            samples = pd.read_csv(dir_input + "/Samples_" + str(b) + "_" + t + ".csv", index_col=0)
            temp = pd.concat([temp, samples])
        except:
            continue
    return temp

In [128]:
allSamples = mergeBatches(batches,"/Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma","A")

In [129]:
allSamples

,p4961_POS,p2625_POS,p388_POS,p3906_POS,L-beta-aspartyl-L-glycine_POS,p4608_POS,n3565_NEG,n3217_NEG,n1421_NEG,p5485_POS,...,n2032_NEG,p2868_POS,n2856_NEG,p4778_POS,p1779_POS,p4902_POS,group,subgroup,gestational_age,gestational_age_at_collection
DP3-0126 A,14.684652,16.749745,23.318046,13.448339,18.634865,14.375142,12.153222,16.543156,20.680224,13.334759,...,19.180957,19.090672,17.382545,13.572060,20.026277,14.868762,Control,Control,38.714286,11.571429
DP3-0049E A,14.217743,16.589084,23.082079,13.338005,18.230664,13.835833,12.184757,15.276652,20.678922,12.979446,...,18.802536,19.327790,17.138188,13.752171,20.477195,14.148186,FGR,FGR <3,36.428571,10.571429
DP3-0080 A,14.552240,18.184424,23.450324,13.311859,17.763124,14.344859,12.163992,16.465158,20.311723,13.289870,...,19.006359,18.905107,17.260853,13.418998,20.044280,13.941387,HDP,EO severe PE,33.600000,11.428571
DP3-0130E A,14.440003,16.711755,22.921763,13.448531,18.191152,13.915124,12.214486,16.154708,20.696160,13.059334,...,18.837432,18.721994,17.668576,13.312601,19.995764,14.101903,Control,Control,40.142857,12.142857
DP3-0094 A,14.322486,16.548184,23.177650,13.271700,17.616737,13.905682,12.129728,15.761415,20.721160,12.933958,...,19.102677,19.462113,17.602385,13.391431,20.671501,14.046611,Control,Control,40.000000,12.142857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DP3-0358EA,13.944880,16.617202,23.344189,13.413851,18.085886,13.859183,12.105188,16.672700,20.862503,13.589636,...,19.039601,18.508434,16.836961,14.277347,20.473691,12.201831,HDP,LO severe PE,35.000000,13.428571
DP3-0374A,13.166090,16.951993,23.635916,13.332738,18.426718,14.581818,12.222377,16.812617,20.770784,13.334602,...,19.180246,19.809997,17.621400,14.732288,20.408069,14.491102,FGR,FGR <5,38.000000,13.285714
DP3-0287A,14.058740,16.551634,23.503225,13.261429,17.223285,13.345333,12.112965,16.344661,20.751960,13.001661,...,18.815076,19.665581,17.524221,14.679156,20.492580,14.311979,Control,Control,39.571429,13.571429
DP3-0256A,14.181557,15.675376,23.173188,13.289933,17.421613,13.531881,12.156213,16.593976,20.753445,13.476202,...,18.881860,19.684686,17.046957,14.323199,20.645868,14.814409,Control,Control,40.000000,11.714286


In [130]:
x

,med,mad,tissue,timepoint,datatype,analyte_ID
Amoxicillin_POS,13.341908,0.135476,plasma,A,metabolites,Amoxicillin_POS
p487_POS,23.117087,0.465887,plasma,A,metabolites,p487_POS
n304_NEG,22.091415,0.258679,plasma,A,metabolites,n304_NEG
p3014_POS,19.410243,0.356406,plasma,A,metabolites,p3014_POS
n694_NEG,20.831865,0.153171,plasma,A,metabolites,n694_NEG
...,...,...,...,...,...,...
p5751_POS,15.351127,0.724511,plasma,A,metabolites,p5751_POS
p3224_POS,19.309842,0.385820,plasma,A,metabolites,p3224_POS
p3083_POS,18.495527,0.375046,plasma,A,metabolites,p3083_POS
p411_POS,24.532796,0.246376,plasma,A,metabolites,p411_POS


In [133]:
# remove NaNs
# check if mad is 0, if is, log
scores = pd.DataFrame(index=allSamples.index)

for m in allSamples.columns[:-1]:
    temp = (allSamples[m] - x.loc[m, "med"]) / x.loc[m, "mad"]
    scores[m] = temp

/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_51152/1045040521.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  scores[m] = temp


KeyError: 'D3-Alanine-ISTD_POS'

In [31]:
scores

,Amoxicillin_POS,p487_POS,n304_NEG,p3014_POS,n694_NEG,p1103_POS,p1187_POS,n3543_NEG,p4718_POS,p5146_POS,...,p469_POS,p952_POS,p4677_POS,p2942_POS,p3713_POS,p5751_POS,p3224_POS,p3083_POS,p411_POS,p4768_POS
DP3-0048 A,69137.301165,1.770321e+07,2.130609e+07,1.415473e+06,8.579899e+06,185647.988824,1.795936e+06,68817.999792,31658.522557,30771.964180,...,3.705260e+06,199070.475378,70524.492119,205653.751491,301328.356437,28170.604684,1.303558e+06,6.847211e+05,9.858317e+07,4988.317598
DP3-0071 A,74160.411132,3.384624e+07,2.126305e+07,1.441892e+06,1.354674e+07,177717.707797,2.717730e+06,69622.790413,34632.070015,43428.915122,...,3.300950e+06,190567.502706,97109.048276,223019.374646,332770.208288,99160.914447,1.436528e+06,8.787352e+05,1.172060e+08,5924.440005
DP3-0124 A,69532.025718,2.020624e+07,1.933590e+07,1.877530e+06,1.220338e+07,262522.297800,1.975175e+06,68619.451384,26428.609720,37847.844962,...,2.919169e+06,281496.324619,126825.746782,247508.674824,359715.203909,92420.034226,1.448548e+06,1.434145e+06,1.043550e+08,9417.707965
DP3-0018 A,69310.224965,1.689133e+07,1.408149e+07,1.885336e+06,1.766883e+07,160239.611515,3.424615e+06,68799.844279,30418.988550,37113.039712,...,3.352906e+06,171827.211594,67381.114762,253119.014640,303755.899708,46269.288413,1.431394e+06,1.852601e+06,1.187816e+08,2662.378535
DP3-0037 A,69259.244763,2.345421e+07,2.002163e+07,1.870356e+06,1.513324e+07,200674.941252,1.936389e+06,70347.113760,28963.885860,45162.274549,...,3.021885e+06,215182.611302,104221.972663,257440.714032,409848.959652,56028.776765,1.344794e+06,2.715137e+05,1.085993e+08,6527.500459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DP3-0259A,-1.354862,-1.859422e+00,-2.683438e-01,3.106387e-01,-1.364522e+00,0.243680,-9.021799e-02,-0.052872,0.457598,-0.688774,...,-2.949984e+00,0.285187,0.466589,-1.726435,0.064879,-1.226023,3.622515e-01,-1.017507e+00,6.911198e-01,-0.526905
DP3-0387A,-1.824665,-1.291736e-01,3.489109e-01,-1.200858e+00,-6.885499e-01,-0.282712,-3.089097e-01,0.556242,0.919446,-0.534070,...,-6.378729e-01,-0.487663,-1.000000,-1.442962,-1.632721,-0.268143,-1.266469e+00,-8.715357e-01,-2.375504e-01,-0.116333
DP3-0279A,0.881315,1.059300e-01,3.470259e-01,-6.305717e-01,-2.784036e-01,0.549304,-1.323203e+00,82.789969,-0.375614,0.894286,...,-2.795542e+00,0.416070,1.312618,-0.389800,1.010345,-0.136098,0.000000e+00,-2.685270e+00,-2.866353e-01,-0.386355
DP3-0392A,-0.206791,4.996994e-01,4.486941e-01,5.041866e-01,-4.328502e-01,-0.851826,1.053990e+00,-2.472157,7.348614,-0.315311,...,1.651954e-01,-0.517263,1.262330,-0.401374,0.453009,0.200399,5.326817e-01,-1.488595e-01,-4.188225e-01,-0.243035


In [172]:
meta = pd.read_excel("/Users/kaylaxu/Desktop/data/raw_data/dp3 master table v2.xlsx", index_col=1, sheet_name="n=133 metabolomics")

In [173]:
clean_meta = meta[meta.index.notna()]

In [174]:
list(clean_meta.loc[clean_meta["Sample ID"] == "DP3-0165EA", "group"])[0]

KeyError: 'Sample ID'

In [119]:
scores.map(lambda x: 1 if x > 3 else (-1 if x < -3 else 0)).sum(axis=1)

DP3-0048 A    1885
DP3-0071 A    1885
DP3-0124 A    1885
DP3-0018 A    1885
DP3-0037 A    1885
              ... 
DP3-0259A      -12
DP3-0387A      -10
DP3-0279A       -7
DP3-0392A       20
DP3-0281EA       3
Length: 134, dtype: int64

In [138]:
list(meta.loc[meta["Sample ID"] == "DP3-0005 B", :].index)[0]

'DP3-0005'

In [149]:
p = "DP3-0034"
t = "A"
m = "N-(1,3-benzodioxol-5-yl)-2-methyl-5-(piperidinosulfonyl)-3-furamide_POS"
outlierMatrix = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/outlier_flags_matrix_plasma_A_MTBL.csv", index_col=0)

In [150]:
outlierMatrix

,D3-Creatinine-ISTD_POS,p3823_POS,p1690_POS,p5795_POS,p2942_POS,"N-(1,3-benzodioxol-5-yl)-2-methyl-5-(piperidinosulfonyl)-3-furamide_POS","N1-(alpha-D-ribosyl)-5,6-dimethyl-benzimidazole_POS",p3174_POS,Methoxsalen_POS,n3688_NEG,...,n840_NEG,N-Ethylacetamide_POS,D-Glycero-D-galacto-heptitol_NEG,n1923_NEG,n2403_NEG,n3021_NEG,p2143_POS,2-Pentenedial_NEG,p2785_POS,n3442_NEG
DP3-0111 A,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
DP3-0121 A,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,1,0,0,0,0,0
DP3-0034 A,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
DP3-0140E A,0,0,0,0,-1,0,0,0,0,0,...,0,-1,0,0,0,0,1,0,0,0
DP3-0126 A,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,1,0,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DP3-0266A,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,-1,0
DP3-0346A,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,-1,0
DP3-0388A,0,1,0,0,0,0,0,1,-1,0,...,0,-1,0,0,0,0,-1,0,1,0
DP3-0281EA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [152]:
list(outlierMatrix.loc[[p in x for x in outlierMatrix.index], m])[0]

1

In [58]:
persistentMatrix = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/persistent_outliers_plasma_MTBL.csv", index_col=0)

In [7]:
temp = persistentMatrix.loc[persistentMatrix["group"] != "Control",:]

In [8]:
from collections import Counter

In [21]:
persistentMatrix.loc[persistentMatrix["analyte_ID"] == "p1647_POS",]

,patient_ID,analyte_ID,group,subgroup,total_timepoints,outlier_timepoint_count,outlier_direction,outlier_timepoints,outlier_mad_scores,mean_outlier_mad
141,DP3-0034,p1647_POS,Control,Control,4,3,elevated,"['A', 'B', 'C']","[4.36137660316406, 3.3107691837752173, 4.05638...",3.909510
1265,DP3-0045,p1647_POS,HDP,FGR <5+EO mild PE,2,2,elevated,"['A', 'B']","[24.795581573491223, 23.4914826882095]",24.143532
2225,DP3-0025,p1647_POS,HDP,LO severe PE,4,3,elevated,"['A', 'B', 'C']","[4.248105550939726, 4.547115578102822, 3.75718...",4.184135
3053,DP3-0214,p1647_POS,sPTB,sPTB,3,3,elevated,"['B', 'C', 'D']","[31.435308864906272, 38.45921191314632, 23.367...",31.087200
3149,DP3-0197,p1647_POS,HDP,EO severe PE,3,3,elevated,"['A', 'B', 'C']","[29.618516782658883, 25.067290576615893, 36.64...",30.442615
6381,DP3-0399,p1647_POS,FGR,FGR <3,5,5,elevated,"['A', 'B', 'C', 'D', 'E']","[24.025467321102866, 14.154303689783696, 36.37...",27.495435
9705,DP3-0423E,p1647_POS,HDP,LO mild SIPE,5,5,elevated,"['A', 'B', 'C', 'D', 'E']","[25.292419779934527, 26.038184812623292, 31.23...",33.123685


In [10]:
import ast

In [19]:
ast.literal_eval(persistentMatrix.loc[persistentMatrix["patient_ID"] == "DP3-0111", "outlier_timepoints"][0])

['A', 'D']

In [27]:
len(set(persistentMatrix.loc[["A" in x or "B" in x for x in persistentMatrix["outlier_timepoints"]],"patient_ID"]))

131

In [32]:
m = "p1647_POS"
complicationOnly = persistentMatrix.loc[persistentMatrix["analyte_ID"] == m,:]

In [36]:
hasA = complicationOnly.loc[["A" in x for x in complicationOnly["outlier_timepoints"]],:]
hasB = complicationOnly.loc[["B" in x for x in complicationOnly["outlier_timepoints"]],:]
#early = hasA.union(hasB)
#both = hasA & hasB

In [37]:
hasA

,patient_ID,analyte_ID,group,subgroup,total_timepoints,outlier_timepoint_count,outlier_direction,outlier_timepoints,outlier_mad_scores,mean_outlier_mad
141,DP3-0034,p1647_POS,Control,Control,4,3,elevated,"['A', 'B', 'C']","[4.36137660316406, 3.3107691837752173, 4.05638...",3.909510
1265,DP3-0045,p1647_POS,HDP,FGR <5+EO mild PE,2,2,elevated,"['A', 'B']","[24.795581573491223, 23.4914826882095]",24.143532
2225,DP3-0025,p1647_POS,HDP,LO severe PE,4,3,elevated,"['A', 'B', 'C']","[4.248105550939726, 4.547115578102822, 3.75718...",4.184135
3149,DP3-0197,p1647_POS,HDP,EO severe PE,3,3,elevated,"['A', 'B', 'C']","[29.618516782658883, 25.067290576615893, 36.64...",30.442615
6381,DP3-0399,p1647_POS,FGR,FGR <3,5,5,elevated,"['A', 'B', 'C', 'D', 'E']","[24.025467321102866, 14.154303689783696, 36.37...",27.495435
9705,DP3-0423E,p1647_POS,HDP,LO mild SIPE,5,5,elevated,"['A', 'B', 'C', 'D', 'E']","[25.292419779934527, 26.038184812623292, 31.23...",33.123685


In [38]:
hasB

,patient_ID,analyte_ID,group,subgroup,total_timepoints,outlier_timepoint_count,outlier_direction,outlier_timepoints,outlier_mad_scores,mean_outlier_mad
141,DP3-0034,p1647_POS,Control,Control,4,3,elevated,"['A', 'B', 'C']","[4.36137660316406, 3.3107691837752173, 4.05638...",3.909510
1265,DP3-0045,p1647_POS,HDP,FGR <5+EO mild PE,2,2,elevated,"['A', 'B']","[24.795581573491223, 23.4914826882095]",24.143532
2225,DP3-0025,p1647_POS,HDP,LO severe PE,4,3,elevated,"['A', 'B', 'C']","[4.248105550939726, 4.547115578102822, 3.75718...",4.184135
3053,DP3-0214,p1647_POS,sPTB,sPTB,3,3,elevated,"['B', 'C', 'D']","[31.435308864906272, 38.45921191314632, 23.367...",31.087200
3149,DP3-0197,p1647_POS,HDP,EO severe PE,3,3,elevated,"['A', 'B', 'C']","[29.618516782658883, 25.067290576615893, 36.64...",30.442615
6381,DP3-0399,p1647_POS,FGR,FGR <3,5,5,elevated,"['A', 'B', 'C', 'D', 'E']","[24.025467321102866, 14.154303689783696, 36.37...",27.495435
9705,DP3-0423E,p1647_POS,HDP,LO mild SIPE,5,5,elevated,"['A', 'B', 'C', 'D', 'E']","[25.292419779934527, 26.038184812623292, 31.23...",33.123685


In [45]:
early = hasA.merge(hasB, how="outer")
both = hasA.merge(hasB, how="inner")


In [48]:
early["mean_outlier_mad"].sum()

np.float64(154.3861118237865)

In [50]:
len(both.index)

6

In [51]:
dict = {"a": 1, "b": 2, "c":3}

In [58]:
temp = list(dict.keys())

In [61]:
temp.remove("a")

In [62]:
temp

['b', 'c']

In [66]:
temp = [1,2,3]
y = 1

In [70]:
complicationOnly

,patient_ID,analyte_ID,group,subgroup,total_timepoints,outlier_timepoint_count,outlier_direction,outlier_timepoints,outlier_mad_scores,mean_outlier_mad
141,DP3-0034,p1647_POS,Control,Control,4,3,elevated,"['A', 'B', 'C']","[4.36137660316406, 3.3107691837752173, 4.05638...",3.909510
1265,DP3-0045,p1647_POS,HDP,FGR <5+EO mild PE,2,2,elevated,"['A', 'B']","[24.795581573491223, 23.4914826882095]",24.143532
2225,DP3-0025,p1647_POS,HDP,LO severe PE,4,3,elevated,"['A', 'B', 'C']","[4.248105550939726, 4.547115578102822, 3.75718...",4.184135
3053,DP3-0214,p1647_POS,sPTB,sPTB,3,3,elevated,"['B', 'C', 'D']","[31.435308864906272, 38.45921191314632, 23.367...",31.087200
3149,DP3-0197,p1647_POS,HDP,EO severe PE,3,3,elevated,"['A', 'B', 'C']","[29.618516782658883, 25.067290576615893, 36.64...",30.442615
6381,DP3-0399,p1647_POS,FGR,FGR <3,5,5,elevated,"['A', 'B', 'C', 'D', 'E']","[24.025467321102866, 14.154303689783696, 36.37...",27.495435
9705,DP3-0423E,p1647_POS,HDP,LO mild SIPE,5,5,elevated,"['A', 'B', 'C', 'D', 'E']","[25.292419779934527, 26.038184812623292, 31.23...",33.123685


In [74]:
all_mad_scores = []
for i in complicationOnly.index:
    all_mad_scores.extend(ast.literal_eval(complicationOnly.loc[i,"outlier_mad_scores"]))

In [ ]:
mean_outlier_mads = list(complicationOnly["mean_outlier_mad"])

In [84]:
list(mean_outlier_mads)

[3.909509875974494,
 24.14353213085036,
 4.184135355275778,
 31.08719962543532,
 30.44261491831252,
 27.495435105188022,
 33.12368481275001]

In [86]:
complicationOnly.iloc[0,:]["patient_ID"]

'DP3-0034'

In [9]:
a = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/biomarker_complication_specific_plasma.csv", index_col=0)
b = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/biomarker_early_warning_plasma.csv", index_col=0)
c = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/biomarker_most_extreme_plasma.csv", index_col=0)
d = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/biomarker_most_persistent_plasma.csv", index_col=0)
e = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/biomarker_most_prevalent_plasma.csv", index_col=0)

In [ ]:
df = pd.DataFrame(0, index=a1.columns, columns=["most_persistent", "most_prevalent", "early_warning", "complication_specific", "most_extreme"])


In [20]:
df = df.drop(index=["patient_ID", "group", "subgroup", "gestational_age", "gestational_age_at_collection"])

In [ ]:
for m in df.index:
    if m in set(a["analyte_ID"]):
        df.loc[m, "complication_specific"] = 1
    if m in set(b[])


p4375_POS
p3858_POS
4-Thialysine_POS
1-(2-Furylmethyl)-5-oxopyrrolidine-3-carboxylic acid_NEG
4-Nitrophenyl beta-D-xyloside_POS
p3532_POS
Tetramethylene sulfoxide_POS
p4868_POS
p2333_POS
n234_NEG
p5097_POS
3-(dimethylamino)-1-(2-pyridyl)prop-2-en-1-one_POS
p515_POS


In [30]:
'1-(2-Furylmethyl)-5-oxopyrrolidine-3-carboxylic acid_NEG' in set(a["analyte_ID"])

True

In [29]:
set(a["analyte_ID"])

{'1-(2-Furylmethyl)-5-oxopyrrolidine-3-carboxylic acid_NEG',
 '3-(dimethylamino)-1-(2-pyridyl)prop-2-en-1-one_POS',
 '4-Nitrophenyl beta-D-xyloside_POS',
 '4-Thialysine_POS',
 'Tetramethylene sulfoxide_POS',
 'n234_NEG',
 'p2333_POS',
 'p3532_POS',
 'p3858_POS',
 'p4375_POS',
 'p4868_POS',
 'p5097_POS',
 'p515_POS'}

In [53]:
df = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/biomarker_summary_crosswalk_plasma.csv", index_col=0)


In [54]:
df.loc[df["super_candidate"],:]

,most_persistent,most_prevalent,early_warning,complication_specific,most_extreme,super_candidate
p5015_POS,1,1,1,0,0,True
p3469_POS,0,1,1,0,1,True
p4375_POS,0,1,1,1,0,True
p1452_POS,1,1,1,0,0,True
p2292_POS,1,1,1,0,0,True
p3468_POS,0,1,1,0,1,True
"3-(2-methylpropyl)-octahydropyrrolo[1,2-a]pyrazine-1,4-dione_POS",1,1,1,0,0,True
4-Thialysine_POS,0,1,1,1,0,True
1-(2-Furylmethyl)-5-oxopyrrolidine-3-carboxylic acid_NEG,0,1,1,1,0,True
4-Nitrophenyl beta-D-xyloside_POS,1,1,1,1,0,True


In [57]:
a1

,D3-Creatinine-ISTD_POS,p3823_POS,p1690_POS,p5795_POS,p2942_POS,"N-(1,3-benzodioxol-5-yl)-2-methyl-5-(piperidinosulfonyl)-3-furamide_POS","N1-(alpha-D-ribosyl)-5,6-dimethyl-benzimidazole_POS",p3174_POS,Methoxsalen_POS,n3688_NEG,...,n3021_NEG,p2143_POS,2-Pentenedial_NEG,p2785_POS,n3442_NEG,patient_ID,group,subgroup,gestational_age,gestational_age_at_collection
DP3-0068A,24.539903,13.532211,21.540364,14.024703,14.761840,13.666557,15.534550,15.488050,14.814957,11.939747,...,16.608155,18.622838,22.822123,13.775127,12.564474,DP3-0068,FGR,LO FGR <5,38.571429,12.142857
DP3-0152A,24.607585,14.209991,21.707366,14.088314,14.689568,13.764620,16.543768,15.483095,14.832507,11.972566,...,17.678873,18.605813,23.593900,13.796759,12.650198,DP3-0152,Control,Control,40.000000,11.857143
DP3-0166A,24.636039,12.716396,21.165903,13.979405,14.627921,13.484160,16.234793,15.352370,15.013048,11.909662,...,17.319043,17.091345,23.269763,13.699800,12.417734,DP3-0166,Control,Control,39.714300,13.000000
DP3-0161A,24.630525,13.329226,21.463657,13.757993,14.716635,13.599670,17.179500,15.334558,14.874019,11.987126,...,17.158561,18.585834,23.060163,13.730143,12.566932,DP3-0161,FGR,LO FGR <5,38.000000,11.714286
DP3-0207A,24.669245,14.566476,21.369441,14.083710,14.758607,13.518625,16.031825,15.564700,14.980788,11.899113,...,17.162865,18.614500,23.056907,13.970243,12.539428,DP3-0207,HDP,LO severe PE,37.000000,15.571429
DP3-0214A,24.664062,14.042253,21.606260,14.003858,14.835701,13.764454,17.503668,15.237063,14.879101,11.916979,...,17.409303,18.750641,23.027765,13.659566,12.607172,DP3-0214,sPTB,sPTB,31.571429,11.857143
DP3-0070A,24.565606,14.060614,21.665835,14.062886,14.736988,13.686459,13.433881,15.398837,14.770866,12.790437,...,16.961750,18.566059,23.053957,13.817160,12.579965,DP3-0070,Control,Control,39.571429,8.285714
DP3-0197A,24.447841,13.680254,21.045374,13.611488,14.651957,13.402324,15.807528,15.244152,14.685526,11.933351,...,17.119056,18.525077,22.848578,13.426401,12.337601,DP3-0197,HDP,EO severe PE,34.142900,14.142857
DP3-0142A,24.543989,15.266331,21.795805,14.040889,14.963834,13.660126,16.288259,15.497179,14.894623,12.119731,...,16.782084,18.785420,22.709101,13.810379,12.770425,DP3-0142,Control,Control,37.714286,13.142857
DP3-0116A,24.780964,14.538320,21.517902,14.009288,14.749271,13.616207,15.415671,15.473409,14.850047,11.863828,...,16.434242,18.747275,23.601678,13.744095,12.468125,DP3-0116,Control,Control,40.285714,11.285714


df

In [60]:
mads = {}
mads["A"] = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/mad_scores_matrix_plasma_A_MTBL.csv", index_col=0)
mads["B"] = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/mad_scores_matrix_plasma_B_MTBL.csv", index_col=0)
mads["C"] = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/mad_scores_matrix_plasma_C_MTBL.csv", index_col=0)
mads["D"] = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/mad_scores_matrix_plasma_D_MTBL.csv", index_col=0)
mads["E"] = pd.read_csv("/Users/kaylaxu/Desktop/data/MAD_analyses/mad_scores_matrix_plasma_E_MTBL.csv", index_col=0)

In [61]:
mads

{'A':              D3-Creatinine-ISTD_POS  p3823_POS  p1690_POS  p5795_POS  \
 DP3-0111 A                 0.160193  -2.484321   1.151011  -2.734000   
 DP3-0121 A                 0.855688   2.066832   1.111996   0.624784   
 DP3-0034 A                -2.035729  -0.766768   1.120818   2.888492   
 DP3-0140E A                1.210384  -2.020395   0.036318  -0.931484   
 DP3-0126 A                -0.367375  -2.747865   1.192713  -0.727368   
 ...                             ...        ...        ...        ...   
 DP3-0266A                  0.368014  -2.361713   0.894329   1.029345   
 DP3-0346A                 -1.759694  -2.160885   1.407696  -1.119904   
 DP3-0388A                  1.108487   3.269551   2.404998   2.293107   
 DP3-0281EA                -1.119419  -0.742315   0.318157   1.274751   
 DP3-0305A                 -0.920952   2.362428  -1.007980   1.724927   
 
              p2942_POS  \
 DP3-0111 A    1.020708   
 DP3-0121 A    0.533605   
 DP3-0034 A    2.868474   
 DP3-0140

In [ ]:
def plot_patient_trajectories(data_dict, metabolite_name, control_IDs):
    # Combine dictionary into a single long-format DataFrame
    combined_data = []
    
    for timepoint, df in data_dict.items():
        temp_df = df.copy()
        temp_df['Timepoint'] = timepoint
        
        # Assuming Patient IDs are the index. If they are already a column, skip the reset_index step.
        if 'Patient_ID' not in temp_df.columns:
            temp_df = temp_df.reset_index().rename(columns={'index': 'Patient_ID'})
            
        combined_data.append(temp_df)
        
    long_df = pd.concat(combined_data, ignore_index=True)
    
    # Create a binary condition column (Control vs Complication)
    long_df['Condition'] = np.where(
        long_df['Patient_ID'].isin(control_IDs), 
        'Control', 
        'Complication'
    )
    
    # Set up the plot
    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")
    
    # Plot individual patient lines
    # - `units` tells seaborn which data points belong to the same patient
    # - `estimator=None` stops seaborn from trying to average the lines together
    sns.lineplot(
        data=long_df,
        x='Timepoint',
        y=metabolite_name,
        hue='Condition',
        units='Patient_ID',
        estimator=None,
        palette={'Control': '#4C72B0', 'Complication': '#C44E52'}, # Blue vs Red
        alpha=0.6,    # Make lines slightly transparent to see overlap
        linewidth=1.5
    )
    
    # Formatting
    plt.title(f'Longitudinal Trajectories: {metabolite_name}', fontsize=14, pad=15)
    plt.xlabel('Timepoint', fontsize=12)
    plt.ylabel(f'MAD Score ({metabolite_name})', fontsize=12)
    
    # Ensure timepoints are plotted in exact chronological order
    plt.xticks(ticks=range(len(timepoints)), labels=['A', 'B', 'C', 'D', 'E'])
    
    # Clean up the legend
    plt.legend(title='Group', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    
    plt.show()



In [65]:
plot_patient_trajectories(mads, 'D-Raffinose_POS')

KeyError: 'group'